# Automatic layer separation and meshing: Colab quick start

Runs the pipeline from [Victoria-TechNexus/vesuvius-ply-layers](https://github.com/Victoria-TechNexus/vesuvius-ply-layers) on PHerc. Paris 4. Read the README first for what each
stage does.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. A standard (free) session is enough: one tile takes
about 30 minutes and up to about 9 GB of RAM. Run one cell at a time, and one tile at a time.

## 1. Get the code (into your Google Drive)

The code and every result are kept in your Drive, so nothing is lost if the session disconnects. The first run
downloads the repository; later runs only fetch updates.

If the Drive sign-in fails, use the **Mount Drive** button in the folder pane on the left instead, then run this
cell again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
FOLDER = '/content/drive/MyDrive/vesuvius-ply-layers'
if not os.path.exists(FOLDER):
    !git clone https://github.com/Victoria-TechNexus/vesuvius-ply-layers.git "{FOLDER}"
else:
    !git -C "{FOLDER}" pull
%cd {FOLDER}

## 2. Install what phases 1–9 need

In [ ]:
!pip install -q -r requirements.txt

## 3. Choose the area and the settings (no code needed)

Open the folder pane on the left (the folder icon), go to `drive/MyDrive/vesuvius-ply-layers`, and **double-click
`config.py`**. It opens in an editor; changes save automatically.

- `TILES` in `config.py`: the areas to process. Each tile is a name and its z, y, x ranges at level 2. The six
  tiles of the published results are already there (`t1` to `t6`).
- On another scroll, change `SCROLL`, `SCAN` and `SCROLL_CENTRE_XY` in `config.py`.
- How each stage runs is set in the `SETTINGS` block at the top of `run_pipeline.py`, `run_layers.py`,
  `run_mesh.py` and `run_ink.py`, opened the same way.
- `TUNING.md` lists what to change, by symptom.

## 4. Phases 1–6, one tile at a time

Set the tile name and run both cells. Repeat for each tile. A tile is finished when its folder `ckpt_<tile>/`
holds `manifest.json` (written last) and `layers_<tile>/` holds `layers.json`.

In [ ]:
TILE = 't1'
!python run_pipeline.py {TILE}

In [ ]:
!python run_layers.py {TILE}

## 5. Mesh one layer (phase 7)

List the tiles to join; the first is the reference, and `LAYER` is that layer's number in the first tile (see the
layer table printed by `run_layers.py`, and the picture `layers_<tile>/layers_slices_<tile>.png`). With one tile,
the joining step is skipped.

In [ ]:
TILES = 't1 t2 t3 t4 t5 t6'
LAYER = 11
!python run_mesh.py --tiles {TILES} --layer {LAYER}

## 6. Ink (phases 8–10)

Phase 10 needs the challenge's own ink code and model, which this repository does not include:

1. Install the `vesuvius` package from [ScrollPrize/villa](https://github.com/ScrollPrize/villa) (MIT), following
   its instructions.
2. Download `step-075000.pth` from [scrollprize/ink_9um](https://huggingface.co/scrollprize/ink_9um)
   (`hybrid_3d2d-seed42/`) and put it in this folder.
3. Run the cell below for the system libraries it needs.

In [ ]:
!apt-get -qq install -y libblosc-dev
!pip install -q imagecodecs albumentations

`MESH` is the folder `run_mesh.py` printed at the end (`mesh_<tiles>_layerNN`). `ID` is any number; it names the
output files. Use `--device cuda` on a GPU session.

In [ ]:
MESH = './mesh_t1-t2-t3-t4-t5-t6_layer11'
ID = 9611
!python run_ink.py --name layer{LAYER} --id {ID} --ckpt {MESH} --wraps {MESH}/wraps --device cuda

The pictures are in `diagnostics/`: `ink_<id>.png` (flattened surface and ink) and `grid_<id>_stretch.png`.